In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, StringType
from itertools import chain
from pyspark.sql import Window

#Reading From Silver Table

In [0]:
df = spark.table("workspace.silver.promo_calander")

#Transform

##Create surrogate key

In [0]:
w = Window.orderBy(F.monotonically_increasing_id())
df = df.withColumn("promo_key", F.row_number().over(w))
df = df.select("promo_key", *[c for c in df.columns if c != "promo_key"])

In [0]:
df.display()

#Write into gold table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_promo_calander")
)

In [0]:
%sql
SELECT * FROM gold.dim_promo_calander